# Claude Certified Architect — Foundations
## Domain 2: Tool Design & MCP Integration
**Exam weight: 18%**

Every reliability problem in an agentic system either traces back to a poorly designed tool interface or to wrong tool selection — and the two are related. This domain covers the full stack of tool design decisions: how descriptions route calls correctly, how structured errors enable automated recovery, how scoped access prevents agents from using tools outside their role, how MCP servers extend capabilities at the team level, and how the built-in Claude Code tools map to specific codebase navigation tasks.

This notebook covers all five task statements in Domain 2. Each section includes:
- What the concept means in practice
- Runnable code demonstrating the skill
- Anti-patterns alongside correct patterns where relevant

**Prerequisites:** `pip install anthropic`  
**Auth:** Set `ANTHROPIC_API_KEY` as an environment variable before running.

### Task Statements Covered
- **2.1** Design effective tool interfaces with clear descriptions and boundaries
- **2.2** Implement structured error responses for MCP tools
- **2.3** Distribute tools appropriately across agents and configure tool choice
- **2.4** Integrate MCP servers into Claude Code and agent workflows
- **2.5** Select and apply built-in tools (Read, Write, Edit, Bash, Grep, Glob) effectively

In [15]:
import anthropic
import json
import re

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"
print("Client ready.")

Client ready.


---
## Task Statement 2.1: Design effective tool interfaces with clear descriptions and boundaries

Tool descriptions are the routing table your agent uses to decide which tool to call — not the system prompt, not the tool name. The investment in a well-written description pays off across every agent that uses the tool: moving selection accuracy from 60% to 95%+ is usually a matter of three additional sentences.

**What this means in practice:** Tool descriptions are the routing table your agent uses for every tool call decision. A description that says "Analyzes content" gives the model nothing to differentiate it from "Analyzes a document." Include: what the tool does, what inputs it expects, what it returns, example queries that should trigger it, and — crucially — when NOT to use it (the boundary condition that separates it from similar tools).

**Why it matters for an architect:** Tool selection unreliability is one of the most common production failures in agentic systems, and one of the cheapest to fix. Expanding a description from 3 words to 3 sentences often moves selection accuracy from 60% to 95%+. This is the highest-leverage prompt work an architect does — it lives in the tool definition, not the system prompt, and it benefits every agent that uses the tool.


**Core concept:** Tool descriptions are the **primary mechanism** LLMs use for tool selection. When descriptions are minimal or overlapping, the model lacks context to differentiate between similar tools — leading to misrouting.

**Key principles:**
- Include input formats, example queries, edge cases, and boundary explanations
- Eliminate functional overlap between tool names and descriptions
- Split generic tools into purpose-specific tools with defined input/output contracts
- Review system prompts for keyword-sensitive instructions that may create unintended tool associations

**Anti-pattern:** Two tools with near-identical minimal descriptions (e.g., `analyze_content` vs `analyze_document`) cause unreliable selection.

In [16]:
# ANTI-PATTERN: Minimal, overlapping tool descriptions
# These descriptions give the model almost no signal to differentiate the tools.

TOOLS_POOR_DESCRIPTIONS = [
    {
        "name": "analyze_content",
        "description": "Analyzes content.",  # BAD: vague, no input/output contract
        "input_schema": {
            "type": "object",
            "properties": {
                "input": {"type": "string"}
            },
            "required": ["input"]
        }
    },
    {
        "name": "analyze_document",
        "description": "Analyzes a document.",  # BAD: functionally identical description
        "input_schema": {
            "type": "object",
            "properties": {
                "input": {"type": "string"}
            },
            "required": ["input"]
        }
    }
]

def test_tool_selection(tools: list, query: str, label: str) -> str:
    """Call the API with a query and report which tool the model selects."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        tools=tools,
        messages=[{"role": "user", "content": query}]
    )
    selected = None
    for block in response.content:
        if block.type == "tool_use":
            selected = block.name
    print(f"[{label}] Query: '{query}'")
    print(f"  Selected tool: {selected or '(no tool called — text response)'}")
    return selected

print("=== ANTI-PATTERN: Poor tool descriptions ===")
# The model has to guess — selection will be inconsistent
test_tool_selection(
    TOOLS_POOR_DESCRIPTIONS,
    "Search this webpage for mentions of pricing: https://example.com",
    "Poor descriptions"
)
test_tool_selection(
    TOOLS_POOR_DESCRIPTIONS,
    "Extract the key findings from this PDF report",
    "Poor descriptions"
)

=== ANTI-PATTERN: Poor tool descriptions ===
[Poor descriptions] Query: 'Search this webpage for mentions of pricing: https://example.com'
  Selected tool: (no tool called — text response)
[Poor descriptions] Query: 'Extract the key findings from this PDF report'
  Selected tool: (no tool called — text response)


In [17]:
# CORRECT PATTERN: Well-differentiated tool descriptions
# Each description specifies:
# - What it does and when to use it
# - What inputs it expects
# - What it returns
# - When NOT to use it (boundary condition)

TOOLS_GOOD_DESCRIPTIONS = [
    {
        "name": "extract_web_results",
        "description": (
            "Fetches and extracts text content from a live web URL. "
            "Use this when the user provides a URL (http:// or https://) and wants to search, "
            "summarize, or extract information from a webpage. "
            "Input: a URL string. Returns: extracted page text. "
            "Do NOT use for local files, PDFs, or documents — use extract_document_data instead."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "url": {
                    "type": "string",
                    "description": "A valid http:// or https:// URL to fetch"
                }
            },
            "required": ["url"]
        }
    },
    {
        "name": "extract_document_data",
        "description": (
            "Extracts structured data and key findings from uploaded documents (PDF, DOCX, TXT). "
            "Use this when the user refers to an uploaded file, a report, a paper, or any local document. "
            "Input: document filename or path. Returns: extracted text, tables, and key sections. "
            "Do NOT use for live web URLs — use extract_web_results instead."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Filename or path of the uploaded document (e.g., report.pdf, data.docx)"
                }
            },
            "required": ["filename"]
        }
    }
]

print("=== CORRECT PATTERN: Well-differentiated tool descriptions ===")
test_tool_selection(
    TOOLS_GOOD_DESCRIPTIONS,
    "Search this webpage for mentions of pricing: https://example.com",
    "Good descriptions"
)
test_tool_selection(
    TOOLS_GOOD_DESCRIPTIONS,
    "Extract the key findings from this PDF report: annual_report.pdf",
    "Good descriptions"
)

print("\nOBSERVE: With clear descriptions, the model reliably routes to the correct tool.")
print("With minimal descriptions, selection is unpredictable.")

=== CORRECT PATTERN: Well-differentiated tool descriptions ===
[Good descriptions] Query: 'Search this webpage for mentions of pricing: https://example.com'
  Selected tool: extract_web_results
[Good descriptions] Query: 'Extract the key findings from this PDF report: annual_report.pdf'
  Selected tool: extract_document_data

OBSERVE: With clear descriptions, the model reliably routes to the correct tool.
With minimal descriptions, selection is unpredictable.


In [18]:
# Demonstrating: splitting a generic tool into purpose-specific tools
# with defined input/output contracts.
#
# Generic: analyze_document (does everything, selected unreliably)
# Specific: extract_data_points, summarize_content, verify_claim_against_source

TOOLS_SPLIT = [
    {
        "name": "extract_data_points",
        "description": (
            "Extracts specific structured data points from a document: numbers, dates, names, "
            "percentages, measurements. Use when you need specific facts or figures from a document. "
            "Returns a structured list of data point objects with value, unit, and context."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "document_id": {"type": "string"},
                "data_types": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Types to extract, e.g. ['dates', 'percentages', 'currency_amounts']"
                }
            },
            "required": ["document_id"]
        }
    },
    {
        "name": "summarize_content",
        "description": (
            "Generates a prose summary of a document's main arguments, conclusions, or narrative. "
            "Use when the user wants an overview or executive summary, not specific data points. "
            "Do NOT use this to verify specific claims — use verify_claim_against_source instead."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "document_id": {"type": "string"},
                "max_sentences": {"type": "integer", "description": "Target summary length"}
            },
            "required": ["document_id"]
        }
    },
    {
        "name": "verify_claim_against_source",
        "description": (
            "Checks whether a specific claim or statement is supported, contradicted, or unmentioned "
            "in a document. Use for fact-checking or citation verification. "
            "Input: the claim as a string + document_id. "
            "Returns: supported | contradicted | not_found, plus the relevant excerpt."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "document_id": {"type": "string"},
                "claim": {"type": "string", "description": "The specific claim to verify"}
            },
            "required": ["document_id", "claim"]
        }
    }
]

print("=== Split tools: purpose-specific with defined contracts ===")
test_tool_selection(TOOLS_SPLIT, "Give me an overview of what this report covers: doc123", "Split tools")
test_tool_selection(TOOLS_SPLIT, "Does doc456 support the claim that revenue grew 40% YoY?", "Split tools")
test_tool_selection(TOOLS_SPLIT, "Pull all the revenue figures and dates from doc789", "Split tools")

=== Split tools: purpose-specific with defined contracts ===
[Split tools] Query: 'Give me an overview of what this report covers: doc123'
  Selected tool: summarize_content
[Split tools] Query: 'Does doc456 support the claim that revenue grew 40% YoY?'
  Selected tool: verify_claim_against_source
[Split tools] Query: 'Pull all the revenue figures and dates from doc789'
  Selected tool: extract_data_points


'extract_data_points'

**Key exam facts for 2.1:**
- Tool descriptions are the **primary** selection mechanism — minimal descriptions cause unreliable routing.
- Include: input format, example queries, edge cases, boundary conditions (when NOT to use).
- Overlapping descriptions (same name pattern, same description) cause misrouting even with correct system prompts.
- Split generic tools into purpose-specific ones with defined input/output contracts.
- After fixing descriptions, also audit system prompts for keyword-sensitive instructions that may override well-written descriptions.

---
## Task Statement 2.2: Implement structured error responses for MCP tools

An MCP tool that returns "Operation failed" has told the coordinator nothing about what to do next. Structured errors with categories (transient, validation, business, permission) and explicit retry flags are what allow the coordinator to make intelligent recovery decisions automatically rather than routing every failure to a human.

**What this means in practice:** Every MCP tool must return structured error metadata: an `isError` flag, `errorCategory` (transient/validation/business/permission), an `isRetryable` boolean, and a human-readable message. The coordinator uses this metadata to decide: retry the same query (transient), fix the input and retry (validation), explain to the user (business), or escalate (permission). A valid empty result — the query succeeded but found no matches — is NOT an error and must not be flagged as one.

**Why it matters for an architect:** Generic errors ("Operation failed", empty results returned as success) are the equivalent of swallowing exceptions. The coordinator has no information to make recovery decisions. In a four-subagent research pipeline, one timeout silently returned as empty success degrades output quality with no signal that anything went wrong. Structured errors give you the information needed for intelligent, automated recovery.


**Core concept:** Tools must return structured error metadata — not generic messages — so the agent can make appropriate recovery decisions.

**Error categories:**
- `transient` — timeouts, service unavailability (may retry)
- `validation` — invalid input format (do not retry without fixing input)
- `business` — policy violations (do not retry; explain to user)
- `permission` — access denied (do not retry without auth changes)

**Anti-pattern:** Returning `"Operation failed"` or silently returning empty results prevents the agent from making intelligent recovery decisions.

In [19]:
# Structured error response factory
# Models the MCP isError flag pattern + structured metadata

def make_error_response(
    error_category: str,   # transient | validation | business | permission
    is_retryable: bool,
    message: str,          # Human-readable description for the agent
    user_message: str = None,  # Customer-friendly explanation (for business errors)
    partial_results: dict = None
) -> dict:
    """
    MCP-style structured error response.
    The isError flag signals to the agent that this is a failure, not a valid empty result.
    The errorCategory + isRetryable fields enable intelligent recovery.
    """
    response = {
        "isError": True,                      # MCP isError flag pattern
        "errorCategory": error_category,       # Drives agent recovery strategy
        "isRetryable": is_retryable,           # Prevents wasted retry attempts
        "message": message
    }
    if user_message:
        response["userMessage"] = user_message  # Agent uses this to communicate with customer
    if partial_results:
        response["partialResults"] = partial_results  # Agent can use what succeeded
    return response


# Simulate tool functions that return structured errors

def lookup_order_with_errors(order_id: str, customer_id: str = None) -> dict:
    """Simulates various error conditions with structured responses."""

    # Transient error: database temporarily unavailable
    if order_id == "TIMEOUT":
        return make_error_response(
            error_category="transient",
            is_retryable=True,
            message="Database connection timed out after 5s. Retry may succeed."
        )

    # Validation error: bad input format
    if not re.match(r"^ORD-\d+$", order_id):
        return make_error_response(
            error_category="validation",
            is_retryable=False,  # Retrying with same bad input won't help
            message=f"Invalid order_id format: '{order_id}'. Expected format: ORD-<digits> (e.g., ORD-5512)."
        )

    # Permission error: order belongs to a different customer
    if customer_id and order_id == "ORD-9999":
        return make_error_response(
            error_category="permission",
            is_retryable=False,
            message=f"Order {order_id} does not belong to customer {customer_id}. Access denied.",
            user_message="I wasn't able to find that order on your account. Could you double-check the order number?"
        )

    # Business error: order ineligible for refund (outside return window)
    if order_id == "ORD-0001":
        return make_error_response(
            error_category="business",
            is_retryable=False,  # Policy won't change on retry
            message="Order ORD-0001 is outside the 30-day return window (purchased 45 days ago).",
            user_message="This order was placed more than 30 days ago and is no longer eligible for a return under our standard policy."
        )

    # Success
    return {
        "isError": False,
        "order_id": order_id,
        "amount": 49.99,
        "status": "delivered",
        "eligible_for_return": True
    }


# Demonstrate all four error types
test_cases = [
    ("ORD-5512", "CUST-001"),   # Success
    ("TIMEOUT", "CUST-001"),    # Transient
    ("bad-id", "CUST-001"),     # Validation
    ("ORD-9999", "CUST-001"),   # Permission
    ("ORD-0001", "CUST-001"),   # Business
]

for order_id, customer_id in test_cases:
    result = lookup_order_with_errors(order_id, customer_id)
    print(f"Order: {order_id}")
    print(f"  isError: {result.get('isError')} | category: {result.get('errorCategory', 'N/A')} | retryable: {result.get('isRetryable', 'N/A')}")
    if result.get('isError'):
        print(f"  message: {result.get('message')}")
        if 'userMessage' in result:
            print(f"  userMessage: {result['userMessage']}")
    else:
        print(f"  result: {result}")
    print()

Order: ORD-5512
  isError: False | category: N/A | retryable: N/A
  result: {'isError': False, 'order_id': 'ORD-5512', 'amount': 49.99, 'status': 'delivered', 'eligible_for_return': True}

Order: TIMEOUT
  isError: True | category: transient | retryable: True
  message: Database connection timed out after 5s. Retry may succeed.

Order: bad-id
  isError: True | category: validation | retryable: False
  message: Invalid order_id format: 'bad-id'. Expected format: ORD-<digits> (e.g., ORD-5512).

Order: ORD-9999
  isError: True | category: permission | retryable: False
  message: Order ORD-9999 does not belong to customer CUST-001. Access denied.
  userMessage: I wasn't able to find that order on your account. Could you double-check the order number?

Order: ORD-0001
  isError: True | category: business | retryable: False
  message: Order ORD-0001 is outside the 30-day return window (purchased 45 days ago).
  userMessage: This order was placed more than 30 days ago and is no longer eligibl

In [20]:
# Demonstrating: the critical distinction between
# ACCESS FAILURE (needs retry decision) vs VALID EMPTY RESULT (no matches)
# Confusing these two is a common anti-pattern.

def search_orders(customer_id: str, status_filter: str = None) -> dict:
    """
    Returns either:
    - A valid empty result (customer has no matching orders) — isError: False
    - An access failure (search service down) — isError: True
    These must be distinct so the coordinator can decide whether to retry.
    """
    # Simulate service failure
    if customer_id == "CUST-503":
        return make_error_response(
            error_category="transient",
            is_retryable=True,
            message="Search service returned HTTP 503. Retry after 2 seconds."
        )

    # Valid empty result — the query succeeded, there are just no matches
    # This is NOT an error — returning isError: True here would be wrong
    if customer_id == "CUST-NEW":
        return {
            "isError": False,
            "orders": [],                          # Empty list, not an error
            "message": "No orders found for this customer."  # Informational only
        }

    return {
        "isError": False,
        "orders": [{"order_id": "ORD-5512", "status": "delivered"}]
    }


print("=== Access failure vs valid empty result ===")

result_fail = search_orders("CUST-503")
print(f"Service failure — isError: {result_fail['isError']}, retryable: {result_fail.get('isRetryable')}")
print(f"  Coordinator action: retry after delay")

result_empty = search_orders("CUST-NEW")
print(f"\nEmpty result    — isError: {result_empty['isError']}, orders: {result_empty['orders']}")
print(f"  Coordinator action: tell user they have no orders (do NOT retry)")

result_ok = search_orders("CUST-001")
print(f"\nSuccess         — isError: {result_ok['isError']}, orders: {result_ok['orders']}")

=== Access failure vs valid empty result ===
Service failure — isError: True, retryable: True
  Coordinator action: retry after delay

Empty result    — isError: False, orders: []
  Coordinator action: tell user they have no orders (do NOT retry)

Success         — isError: False, orders: [{'order_id': 'ORD-5512', 'status': 'delivered'}]


In [21]:
# Demonstrating: how the agent uses structured errors to make recovery decisions
# The agent receives the error metadata and decides: retry, explain to user, or escalate.

TOOLS_WITH_ERRORS = [
    {
        "name": "lookup_order",
        "description": (
            "Looks up order details by order ID (format: ORD-<digits>). "
            "Returns order status and eligibility. "
            "May return structured errors: transient (retry), validation (fix input), "
            "business (explain policy), or permission (access denied)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "customer_id": {"type": "string"}
            },
            "required": ["order_id"]
        }
    }
]


def agent_with_error_handling(user_message: str, order_id: str, customer_id: str = None):
    """Agent that uses structured error metadata to drive recovery decisions."""
    messages = [{"role": "user", "content": user_message}]
    system = (
        "You are a support agent. When a tool returns isError: true, check errorCategory: "
        "transient errors may be retried; validation errors mean the input was wrong; "
        "business errors mean policy prevents the action (use userMessage to explain); "
        "permission errors mean access is denied. Never retry non-retryable errors."
    )

    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=512, system=system,
            tools=TOOLS_WITH_ERRORS, messages=messages
        )

        if response.stop_reason == "end_turn":
            return " ".join(b.text for b in response.content if hasattr(b, "text"))

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  Tool call: {block.name}({block.input})")
                    result = lookup_order_with_errors(
                        block.input.get("order_id", ""),
                        block.input.get("customer_id", customer_id)
                    )
                    print(f"  Tool result: {json.dumps(result)}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})


print("=== Business error: agent uses userMessage to explain policy ===")
result = agent_with_error_handling(
    "I'd like to return order ORD-0001 please.",
    order_id="ORD-0001",
    customer_id="CUST-001"
)
print(f"\nAgent response: {result}")

=== Business error: agent uses userMessage to explain policy ===
  Tool call: lookup_order({'order_id': 'ORD-0001'})
  Tool result: {"isError": true, "errorCategory": "business", "isRetryable": false, "message": "Order ORD-0001 is outside the 30-day return window (purchased 45 days ago).", "userMessage": "This order was placed more than 30 days ago and is no longer eligible for a return under our standard policy."}

Agent response: I'm sorry, but unfortunately **order ORD-0001 is not eligible for a return**. This order was placed more than 30 days ago, and our standard return policy only allows returns within **30 days of purchase**.

If you believe there's a special circumstance or would like to explore other options (such as an exchange or store credit), I'd recommend reaching out to our support team directly, as they may be able to assist on a case-by-case basis. Is there anything else I can help you with?


**Key exam facts for 2.2:**
- Always return `isError`, `errorCategory`, `isRetryable`, and a human-readable `message`.
- Four categories: `transient` (retry OK), `validation` (fix input), `business` (explain policy), `permission` (access denied).
- Generic errors like `"Operation failed"` prevent intelligent recovery.
- **Critical distinction:** Access failure (`isError: True`) vs valid empty result (`isError: False, results: []`) — these must be different or the coordinator makes wrong decisions.
- Subagents should handle transient failures locally; only propagate errors they cannot resolve (with partial results and what was attempted).

---
## Task Statement 2.3: Distribute tools appropriately across agents and configure tool choice

Giving an agent access to every tool in the system is equivalent to having no tool organization at all — selection reliability degrades with each additional tool the model must evaluate. Scoping each agent to 4–5 tools relevant to its role, and using `tool_choice` to guarantee specific behavior, are the two primary controls an architect has over tool-call reliability.

**What this means in practice:** Each agent should have 4–5 tools scoped to its role. A synthesis agent with 18 tools — including web search tools it has no business using — will misuse them. `tool_choice: "any"` guarantees the model calls a tool rather than returning conversational text (use when you need structured output). Forced selection (`{"type": "tool", "name": "..."}`) guarantees a specific tool is called (use to enforce step ordering, such as running `extract_metadata` before any enrichment step).

**Why it matters for an architect:** Tool count is a reliability dial. Every additional tool added to an agent increases decision complexity and degrades selection accuracy. Scoped access is also a capability boundary — a synthesis agent cannot accidentally trigger data writes if it lacks write tools. The `tool_choice` configuration is what transforms "the model might call a tool" into "the model will call this specific tool."


**Core concept:** Giving an agent too many tools (e.g., 18 instead of 4-5) degrades tool selection reliability. Agents with tools outside their specialization tend to misuse them. Each agent should have only the tools it needs for its role.

**`tool_choice` options:**
- `"auto"` — model decides whether to call a tool or respond with text
- `"any"` — model must call a tool (any tool), cannot return text-only
- `{"type": "tool", "name": "..."}` — model must call this specific tool

In [22]:
# Demonstrating: tool overprovisioning degrades selection reliability
# A synthesis agent given all 18 tools will attempt web searches it shouldn't.

# All tools available in the system
ALL_SYSTEM_TOOLS = [
    # Web search agent tools
    {"name": "web_search", "description": "Searches the web for current information.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "fetch_url", "description": "Fetches content from a specific URL.",
     "input_schema": {"type": "object", "properties": {"url": {"type": "string"}}, "required": ["url"]}},
    {"name": "search_news", "description": "Searches recent news articles.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    # Document analysis agent tools
    {"name": "extract_data_points", "description": "Extracts structured data from documents.",
     "input_schema": {"type": "object", "properties": {"doc_id": {"type": "string"}}, "required": ["doc_id"]}},
    {"name": "summarize_document", "description": "Summarizes a document's content.",
     "input_schema": {"type": "object", "properties": {"doc_id": {"type": "string"}}, "required": ["doc_id"]}},
    {"name": "extract_citations", "description": "Extracts citations and references from documents.",
     "input_schema": {"type": "object", "properties": {"doc_id": {"type": "string"}}, "required": ["doc_id"]}},
    # Synthesis agent tools (what it should actually have)
    {"name": "merge_findings", "description": "Merges findings from multiple sources into a coherent summary.",
     "input_schema": {"type": "object", "properties": {"findings": {"type": "array", "items": {"type": "string"}}}, "required": ["findings"]}},
    {"name": "verify_fact", "description": "Verifies a specific claim against provided source material.",
     "input_schema": {"type": "object", "properties": {"claim": {"type": "string"}, "source": {"type": "string"}}, "required": ["claim", "source"]}},
    # Report generator tools
    {"name": "format_report", "description": "Formats findings into a structured report.",
     "input_schema": {"type": "object", "properties": {"content": {"type": "string"}}, "required": ["content"]}},
    {"name": "add_citations", "description": "Adds inline citations to a report.",
     "input_schema": {"type": "object", "properties": {"report": {"type": "string"}}, "required": ["report"]}},
]

# Scoped tool sets — each agent gets only what it needs
WEB_SEARCH_AGENT_TOOLS = [t for t in ALL_SYSTEM_TOOLS if t["name"] in ["web_search", "fetch_url", "search_news"]]
DOC_ANALYSIS_AGENT_TOOLS = [t for t in ALL_SYSTEM_TOOLS if t["name"] in ["extract_data_points", "summarize_document", "extract_citations"]]
# Synthesis agent gets its core tools + verify_fact for simple lookups
# (avoids 2-3 round trips through coordinator for 85% of verification cases)
SYNTHESIS_AGENT_TOOLS = [t for t in ALL_SYSTEM_TOOLS if t["name"] in ["merge_findings", "verify_fact"]]
REPORT_AGENT_TOOLS = [t for t in ALL_SYSTEM_TOOLS if t["name"] in ["format_report", "add_citations"]]

print("Tool distribution across agents:")
print(f"  Web Search Agent:   {[t['name'] for t in WEB_SEARCH_AGENT_TOOLS]}")
print(f"  Doc Analysis Agent: {[t['name'] for t in DOC_ANALYSIS_AGENT_TOOLS]}")
print(f"  Synthesis Agent:    {[t['name'] for t in SYNTHESIS_AGENT_TOOLS]} (verify_fact = scoped cross-role tool)")
print(f"  Report Agent:       {[t['name'] for t in REPORT_AGENT_TOOLS]}")
print(f"\nTotal tools in system: {len(ALL_SYSTEM_TOOLS)}")
print(f"Max tools any single agent sees: {max(len(WEB_SEARCH_AGENT_TOOLS), len(DOC_ANALYSIS_AGENT_TOOLS), len(SYNTHESIS_AGENT_TOOLS), len(REPORT_AGENT_TOOLS))}")
print("\nOBSERVE: No agent sees more than 3 tools. A synthesis agent with access")
print("to all 10 tools would attempt web searches it has no business making.")

Tool distribution across agents:
  Web Search Agent:   ['web_search', 'fetch_url', 'search_news']
  Doc Analysis Agent: ['extract_data_points', 'summarize_document', 'extract_citations']
  Synthesis Agent:    ['merge_findings', 'verify_fact'] (verify_fact = scoped cross-role tool)
  Report Agent:       ['format_report', 'add_citations']

Total tools in system: 10
Max tools any single agent sees: 3

OBSERVE: No agent sees more than 3 tools. A synthesis agent with access
to all 10 tools would attempt web searches it has no business making.


In [23]:
# Demonstrating: tool_choice configuration options
# auto | any | forced specific tool

DEMO_TOOLS = [
    {
        "name": "extract_metadata",
        "description": "Extracts document metadata: title, author, date, document type.",
        "input_schema": {
            "type": "object",
            "properties": {"document_id": {"type": "string"}},
            "required": ["document_id"]
        }
    },
    {
        "name": "extract_financials",
        "description": "Extracts financial data: revenue, costs, margins from financial documents.",
        "input_schema": {
            "type": "object",
            "properties": {"document_id": {"type": "string"}},
            "required": ["document_id"]
        }
    }
]

def show_tool_choice_behavior(tool_choice_config, label: str):
    """Demonstrates how different tool_choice settings affect model behavior."""
    print(f"\n--- {label} ---")
    try:
        kwargs = {
            "model": MODEL,
            "max_tokens": 256,
            "tools": DEMO_TOOLS,
            "tool_choice": tool_choice_config,
            "messages": [{"role": "user", "content": "Hello, how are you today?"}]  # No tool trigger
        }
        response = client.messages.create(**kwargs)
        tools_called = [b.name for b in response.content if b.type == "tool_use"]
        text_blocks = [b.text for b in response.content if hasattr(b, "text")]
        print(f"  stop_reason: {response.stop_reason}")
        print(f"  Tools called: {tools_called or 'none'}")
        if text_blocks:
            print(f"  Text response: {text_blocks[0][:80]}..." if len(text_blocks[0]) > 80 else f"  Text: {text_blocks[0]}")
    except Exception as e:
        print(f"  Error: {e}")

# tool_choice: auto — model decides (may return text for a casual greeting)
show_tool_choice_behavior({"type": "auto"}, "tool_choice: auto (model may return text)")

# tool_choice: any — model MUST call a tool, cannot return text-only
show_tool_choice_behavior({"type": "any"}, "tool_choice: any (model must call a tool)")

# tool_choice: forced specific tool — model MUST call extract_metadata
show_tool_choice_behavior(
    {"type": "tool", "name": "extract_metadata"},
    "tool_choice: forced (must call extract_metadata)"
)

print("\nOBSERVE:")
print("  auto  -> model may respond with text for casual queries")
print("  any   -> model must call a tool (use to guarantee structured output)")
print("  forced -> model must call the named tool (use to enforce step ordering, e.g., extract_metadata before enrichment)")


--- tool_choice: auto (model may return text) ---
  stop_reason: end_turn
  Tools called: none
  Text response: Hello! I'm doing great, thank you for asking! How can I help you today? Whether ...

--- tool_choice: any (model must call a tool) ---
  stop_reason: tool_use
  Tools called: ['extract_metadata']

--- tool_choice: forced (must call extract_metadata) ---
  stop_reason: tool_use
  Tools called: ['extract_metadata']

OBSERVE:
  auto  -> model may respond with text for casual queries
  any   -> model must call a tool (use to guarantee structured output)
  forced -> model must call the named tool (use to enforce step ordering, e.g., extract_metadata before enrichment)


In [26]:
# Demonstrating: use tool_choice forced to enforce step ordering
# Phase 1: restrict the tool list to extract_metadata only AND force it.
# Restricting the list is essential — tool_choice forced alone allows the model
# to also call other tools that are visible, which breaks phase isolation.
# Phase 2: expand to full tool list with tool_choice: auto for enrichment.

def two_phase_extraction(document_id: str):
    """
    Phase 1: Only extract_metadata is available — cannot call enrichment tools yet.
    Phase 2: Full tool list with auto tool_choice; model decides what enrichment to apply.
    """

    print(f"Phase 1: Forcing extract_metadata for document '{document_id}'")

    # Phase 1: restrict tools to ONLY extract_metadata so the model cannot call
    # extract_financials even if it wants to — plus force the call explicitly.
    phase1_tools = [t for t in DEMO_TOOLS if t["name"] == "extract_metadata"]
    phase1_response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        tools=phase1_tools,
        tool_choice={"type": "tool", "name": "extract_metadata"},  # Forced
        messages=[{"role": "user", "content": f"Process document: {document_id}"}]
    )

    metadata_call = next((b for b in phase1_response.content if b.type == "tool_use"), None)
    print(f"  Forced call: {metadata_call.name if metadata_call else 'ERROR: no tool call'}")

    # Simulate metadata result
    metadata_result = json.dumps({
        "document_id": document_id,
        "type": "financial_report",
        "title": "Q4 2024 Earnings",
        "date": "2024-12-31"
    })

    print(f"  Metadata result: {metadata_result}")
    print(f"Phase 2: Auto tool_choice for enrichment based on metadata")

    # Phase 2: full tool list — model decides what enrichment to apply based on metadata
    phase2_response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        tools=DEMO_TOOLS,
        tool_choice={"type": "auto"},  # Model decides what's needed next
        messages=[
            {"role": "user", "content": f"Process document: {document_id}"},
            {"role": "assistant", "content": phase1_response.content},
            {"role": "user", "content": [{"type": "tool_result", "tool_use_id": metadata_call.id, "content": metadata_result}]}
        ]
    )

    enrichment_calls = [b.name for b in phase2_response.content if b.type == "tool_use"]
    print(f"  Enrichment tools selected: {enrichment_calls or '(text response)'}")
    return enrichment_calls

two_phase_extraction("annual_report_2024")

Phase 1: Forcing extract_metadata for document 'annual_report_2024'
  Forced call: extract_metadata
  Metadata result: {"document_id": "annual_report_2024", "type": "financial_report", "title": "Q4 2024 Earnings", "date": "2024-12-31"}
Phase 2: Auto tool_choice for enrichment based on metadata
  Enrichment tools selected: ['extract_financials']


['extract_financials']

**Key exam facts for 2.3:**
- Too many tools (18 vs 4-5) degrades selection reliability by increasing decision complexity.
- Each agent gets only the tools relevant to its role — prevent cross-specialization misuse.
- Scoped cross-role tools (e.g., `verify_fact` for the synthesis agent) handle 85% of simple cross-role needs without coordinator round trips.
- `tool_choice: "auto"` — model may return text. `"any"` — model must call a tool. `{"type": "tool", "name": "..."}` — forced specific tool.
- Use `tool_choice: "any"` to guarantee structured output. Use forced selection to enforce step ordering.

**`allowedTools` in Claude Code production config:**

In production, tool scoping from cell 13 is enforced by placing an `allowedTools` list in each subagent's `CLAUDE.md` or `.claude/settings.json`. A session that never receives a tool in its allowed list genuinely cannot call it — no prompt instruction required.

```yaml
# search-agent/CLAUDE.md
allowedTools:
  - web_search
  - search_academic_papers
```

```yaml
# synthesis-agent/CLAUDE.md
allowedTools:
  - verify_fact
```

When a coordinator spawns subagents via the `Task` tool, each subagent inherits the `allowedTools` of its launch session unless the subagent's own `CLAUDE.md` defines a more restrictive list. Place the subagent in a subdirectory with its own `CLAUDE.md` to enforce strict per-role scoping.

---
## Task Statement 2.4: Integrate MCP servers into Claude Code and agent workflows

MCP servers extend Claude Code's capabilities with project-specific tools, and the scoping decision — project-level versus user-level — determines whether your teammates get those tools when they clone the repo. The distinction between MCP resources and MCP tools is equally important: resources expose content catalogs at connection time and eliminate the exploratory round-trip calls that tools require.

**What this means in practice:** `.mcp.json` at the project root is committed to version control — all teammates get these MCP server tools when they clone the repo. `~/.claude.json` is personal and never shared. Credentials in `.mcp.json` use `${ENV_VAR}` expansion — the actual token lives in each developer's shell environment, never in the committed file. MCP resources expose read-only content catalogs (open issue summaries, database schemas, documentation indexes) that give the agent visibility into available data without exploratory tool calls.

**Why it matters for an architect:** The project vs. user scoping decision determines whether your team has consistent tooling. Committing credentials is an immediate security incident. Resources vs. tools is an architectural distinction: resources eliminate the 2–3 round-trip tool calls an agent otherwise needs just to discover what data is available — they expose a catalog at connection time so the agent can immediately call the right tool with the right identifier.


**Core concepts:**
- **Project-level** (`.mcp.json`) — shared team tooling, committed to version control
- **User-level** (`~/.claude.json`) — personal or experimental servers, not shared
- Environment variable expansion in `.mcp.json` (e.g., `${GITHUB_TOKEN}`) for credential management without committing secrets
- Tools from all configured MCP servers are discovered at connection time and available simultaneously
- **MCP resources** expose content catalogs (issue summaries, database schemas) to reduce exploratory tool calls
- Use existing community MCP servers for standard integrations (e.g., Jira); reserve custom servers for team-specific workflows

This task statement is primarily configuration knowledge. The cells below show the correct config structures and the rationale for each decision.

**MCP server configuration — the two scopes**

In practice these are JSON files on disk, not Python dicts. The structures below are what Claude Code reads.

**Project-level: `.mcp.json`** — committed to version control, shared with all teammates.

```json
{
  "mcpServers": {
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": {
        "GITHUB_TOKEN": "${GITHUB_TOKEN}"
      }
    },
    "postgres": {
      "command": "python",
      "args": ["-m", "our_mcp_servers.postgres"],
      "env": {
        "DB_CONNECTION_STRING": "${DB_CONNECTION_STRING}"
      }
    }
  }
}
```

Credentials use `${ENV_VAR}` expansion — the actual token lives in each developer's shell environment, never in the committed file.

**User-level: `~/.claude.json`** — personal, never shared or committed.

```json
{
  "mcpServers": {
    "my-experimental-server": {
      "command": "python",
      "args": ["-m", "my_local_experiments.mcp_server"],
      "env": {
        "PERSONAL_API_KEY": "${MY_PERSONAL_API_KEY}"
      }
    }
  }
}
```

**CLI commands for managing MCP servers:**

```bash
# Add a project-level server (writes to .mcp.json — commit this file)
claude mcp add github \
  -e GITHUB_TOKEN \
  -- npx -y @modelcontextprotocol/server-github

# Add a user-level server (writes to ~/.claude.json — personal only)
claude mcp add --scope user my-local-server -- python -m my_server

# Verify what is registered and which scope each server is in
claude mcp list

# Remove a server by name
claude mcp remove my-local-server
```

The `-e GITHUB_TOKEN` flag tells Claude Code to pass that environment variable to the server process. The credential lives in the developer's shell, not in `.mcp.json`.

**Decision table:**

| Situation | Production artifact |
|-----------|---------------------|
| Team-wide tooling everyone should have | `claude mcp add` → committed `.mcp.json` |
| Personal or experimental server | `claude mcp add --scope user` → `~/.claude.json` |
| Credentials | `-e ENV_VAR` flag; actual value in shell environment |
| Verify registration | `claude mcp list` |

**MCP Resources vs. MCP Tools**

- **Tools** execute actions: `web_search`, `lookup_order`, `create_issue`. The model calls them to do work.
- **Resources** expose content catalogs: open issue summaries, database schemas, documentation indexes. They are read-only manifests made visible to the agent at session start, with no exploratory round trips required.

**Without resources:** the agent must call `list_issues()`, `list_tables()`, `list_endpoints()` just to discover what data is available — three exploratory tool calls before any real work begins.

**With resources:** the agent sees a manifest at connection time and can immediately reference the right resource URI (`github://issues/open`) without a discovery call.

Resources surface automatically when a Claude Code session starts. You do not configure a tool call to list them — they appear in the session context.

**Example resource manifest** (returned by the MCP server's `resources/list` endpoint):

```json
{
  "resources": [
    {
      "uri": "github://issues/open",
      "name": "Open Issues Summary",
      "description": "Current open GitHub issues with titles, labels, and assignees. Refreshed every 5 minutes.",
      "mimeType": "application/json"
    },
    {
      "uri": "postgres://schema/public",
      "name": "Database Schema",
      "description": "Public schema table definitions, column names, types, and foreign key relationships.",
      "mimeType": "application/json"
    },
    {
      "uri": "docs://api/v2",
      "name": "API Documentation Index",
      "description": "Index of all API v2 endpoints with descriptions. Use to find the right endpoint before making API calls.",
      "mimeType": "text/markdown"
    }
  ]
}
```

**Key exam point:** Resources reduce the 2–3 round-trip tool calls an agent otherwise needs just to discover what data is available. They expose a catalog at connection time so the agent can immediately call the right tool with the right identifier.

In [27]:
# Demonstrating: enhancing MCP tool descriptions to prevent the agent from
# preferring built-in tools (like Grep) over more capable MCP tools.
#
# If an MCP tool's description is weak, the model will fall back to
# built-in tools it knows well — even if the MCP tool is more powerful.

# WEAK description — agent will prefer Grep over this
WEAK_MCP_TOOL = {
    "name": "semantic_code_search",
    "description": "Searches code.",  # BAD: gives no reason to prefer this over Grep
    "input_schema": {
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"]
    }
}

# STRONG description — agent understands when to prefer this over Grep
STRONG_MCP_TOOL = {
    "name": "semantic_code_search",
    "description": (
        "Performs semantic (meaning-based) search across the entire codebase using embeddings. "
        "PREFER THIS OVER Grep when: searching for concepts rather than exact strings "
        "(e.g., 'authentication logic', 'error handling patterns', 'database connection pooling'). "
        "Returns ranked results with file path, line range, and relevance score. "
        "Use Grep instead when you need exact string or regex matches."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "A natural language description of the code concept to find"
            },
            "top_k": {
                "type": "integer",
                "description": "Number of results to return (default: 10)"
            }
        },
        "required": ["query"]
    }
}

print("Weak MCP tool description:")
print(f"  '{WEAK_MCP_TOOL['description']}'")
print("  -> Agent will likely use Grep instead (it knows Grep well)")
print()
print("Strong MCP tool description:")
print(f"  '{STRONG_MCP_TOOL['description'][:120]}...'")
print("  -> Agent understands when to prefer this tool and when to use Grep instead")
print()
print("KEY: MCP tool descriptions must explicitly explain how the tool compares to")
print("built-in alternatives (Grep, Read, Bash) so the agent knows when to use each.")

Weak MCP tool description:
  'Searches code.'
  -> Agent will likely use Grep instead (it knows Grep well)

Strong MCP tool description:
  'Performs semantic (meaning-based) search across the entire codebase using embeddings. PREFER THIS OVER Grep when: search...'
  -> Agent understands when to prefer this tool and when to use Grep instead

KEY: MCP tool descriptions must explicitly explain how the tool compares to
built-in alternatives (Grep, Read, Bash) so the agent knows when to use each.


**Key exam facts for 2.4:**
- `.mcp.json` (project-level) → shared team tooling, committed to version control
- `~/.claude.json` (user-level) → personal/experimental, NOT shared
- `${ENV_VAR}` expansion in `.mcp.json` — credentials never committed as plaintext
- All configured MCP servers' tools are discovered at connection time and available simultaneously
- MCP resources = content catalogs (read-only manifests); MCP tools = actions
- Enhance MCP tool descriptions to explain capabilities vs built-in alternatives — otherwise the agent defaults to Grep/Read/Bash
- Use community MCP servers for standard integrations (Jira, GitHub); custom servers for team-specific needs

---
## Task Statement 2.5: Select and apply built-in tools (Read, Write, Edit, Bash, Grep, Glob) effectively

The built-in Claude Code tools have sharp distinctions: `Grep` searches file contents while `Glob` finds files by name pattern, `Edit` requires a unique anchor or it fails, and reading every file upfront is the fastest way to exhaust a context budget before any real analysis begins. Knowing which tool to reach for — and what fallback to use when `Edit` fails — is what separates efficient codebase navigation from expensive thrashing.

**What this means in practice:** `Grep` searches file contents (find all callers of `process_refund`, locate every import of `pandas`). `Glob` finds files by name or extension pattern (`**/*.test.tsx`). `Edit` makes targeted changes using a unique text anchor — when the anchor appears multiple times, `Edit` fails and the correct fallback is `Read` + `Write`. Start codebase exploration with `Grep` to find entry points, then follow imports with `Read` — never read all files upfront.

**Why it matters for an architect:** Built-in tool selection is context budget management. Reading a 500-line file with `Read` to find a function that `Grep` would locate in one targeted call wastes hundreds of tokens that could be used for analysis. The `Edit` → `Read`+`Write` fallback matters for CI pipelines: a failed `Edit` that is not caught produces a silent partial modification. Knowing when each built-in tool applies is the difference between an agent that works efficiently at scale and one that exhausts its context window on boilerplate navigation.


**Built-in tool selection guide:**

| Tool | Use when |
|------|----------|
| `Grep` | Searching file **contents** for patterns (function names, error messages, import statements) |
| `Glob` | Finding files by **name or extension patterns** (`**/*.test.tsx`) |
| `Read` | Loading full file contents |
| `Write` | Writing full file contents |
| `Edit` | Targeted modifications using unique text matching |
| `Bash` | Running commands, tests, scripts |

**Key rule:** When `Edit` fails (non-unique text match), fall back to `Read` + `Write`.

**Exploration strategy:** Start with `Grep` to find entry points, then use `Read` to follow imports — do NOT read all files upfront.

In [28]:
# Demonstrating correct tool selection reasoning for built-in tools.
# We simulate the decision logic an agent should apply.

def select_builtin_tool(task: str) -> dict:
    """
    Simulates the tool selection decision an agent should make
    for common codebase tasks.
    """
    response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        system="""
You are an expert at selecting the correct built-in Claude Code tool for codebase tasks.
The available tools are: Grep, Glob, Read, Write, Edit, Bash.

Rules:
- Grep: search file CONTENTS for patterns (function names, strings, imports)
- Glob: find files by NAME or EXTENSION patterns (**/*.test.tsx, src/**/*.py)
- Read: load full file contents
- Write: write full file contents (also fallback when Edit fails)
- Edit: targeted in-place modification using unique text anchor
- Bash: run commands, tests, scripts

Respond with JSON: {"tool": "<name>", "reason": "<one sentence>", "example_usage": "<pseudo-call>"}
""",
        messages=[{"role": "user", "content": f"Task: {task}"}]
    )
    raw = response.content[0].text.strip().replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"tool": "unknown", "reason": raw, "example_usage": ""}


tasks = [
    "Find all files in the project that import 'pandas'",
    "Find all test files in the project (files ending in .test.tsx or .spec.ts)",
    "Load the full contents of src/auth/login.py for review",
    "Fix a typo: change 'recieve' to 'receive' in a function that appears only once in the file",
    "Add a new function to utils.py, but the anchor text appears 3 times in the file",
    "Run the test suite and capture the output",
    "Find where the function 'calculate_refund' is defined and all places it is called",
]

print("Built-in tool selection decisions:")
for task in tasks:
    result = select_builtin_tool(task)
    print(f"\nTask: {task}")
    print(f"  Tool:    {result.get('tool')}")
    print(f"  Reason:  {result.get('reason')}")
    print(f"  Example: {result.get('example_usage')}")

Built-in tool selection decisions:

Task: Find all files in the project that import 'pandas'
  Tool:    Grep
  Reason:  Grep searches file contents for import patterns, making it ideal for finding all files that contain 'import pandas'.
  Example: Grep(pattern='import pandas', path='.')

Task: Find all test files in the project (files ending in .test.tsx or .spec.ts)
  Tool:    Glob
  Reason:  Glob is designed to find files by name or extension patterns, making it perfect for locating test files with specific suffixes.
  Example: glob('**/*.test.tsx', '**/*.spec.ts')

Task: Load the full contents of src/auth/login.py for review
  Tool:    Read
  Reason:  The task requires loading the full contents of a specific file for review.
  Example: Read('src/auth/login.py')

Task: Fix a typo: change 'recieve' to 'receive' in a function that appears only once in the file
  Tool:    Edit
  Reason:  Edit is ideal for targeted in-place modifications using a unique text anchor to fix a specific typo 

**Edit → Read + Write fallback pattern**

`Edit` makes a targeted in-place modification using a text anchor — a unique substring that identifies exactly where in the file the change should land. When the anchor appears more than once, `Edit` fails because it cannot determine which occurrence to modify.

**The rule:** if `Edit` fails due to a non-unique anchor, fall back to `Read` + `Write`.

---

**Anchor uniqueness issue — concrete example**

Suppose a file contains:

```python
def process_payment(amount):
    # TODO: add validation
    return charge(amount)

def process_refund(amount):
    # TODO: add validation
    return refund(amount)

def process_transfer(amount):
    # TODO: add validation
    return transfer(amount)
```

Attempting `Edit` with anchor `    # TODO: add validation` fails because the string appears three times. `Edit` cannot tell which function you intend to modify.

**Fallback: Read then Write**

1. **Read** — load the full file contents into context.
2. **Modify in memory** — make the targeted change with full surrounding context to disambiguate. For example, replace only the `process_refund` block:

```python
def process_refund(amount):
    if amount <= 0:
        raise ValueError('Refund amount must be positive')
    return refund(amount)
```

3. **Write** — write the complete modified file back to disk.

---

**When each approach applies:**

| Situation | Tool |
|-----------|------|
| Anchor text is unique in the file | `Edit` |
| Anchor appears 0 times (not found) | Fix the anchor string, then `Edit` |
| Anchor appears 2+ times | `Read` + `Write` |
| File needs multiple non-adjacent changes | `Read` + `Write` (one pass is cheaper) |

**Why this matters for CI:** a failed `Edit` that is not caught produces a silent partial modification. Always check the return value and fall back explicitly.

**Incremental codebase exploration strategy**

Never read all files upfront. Start with `Grep` to find entry points, then use `Read` to follow imports. Reading every file before knowing which ones are relevant wastes context and buries the signal you need.

---

**Step-by-step example: tracing `process_refund`**

**Step 1 — Grep to locate by content**

```
Grep("process_refund")
→ src/processor.py:8  def process_refund(self, order_id, amount):
→ src/router.py:14    result = self.processor.process_refund(order_id, amount)
```

Two files reference the function. Read only those, not all eight files in the project.

**Step 2 — Read the definition file**

```
Read("src/processor.py")
→ class PaymentProcessor:
      def process_refund(self, order_id, amount):
          return self.repo.create_refund(order_id, amount)
```

The method delegates to `self.repo`, which is a `PaymentRepository`.

**Step 3 — Grep to follow the import**

```
Grep("PaymentRepository")
→ src/processor.py:1   from src.db.repository import PaymentRepository
→ src/db/repository.py:1  class PaymentRepository:
```

Now Read `src/db/repository.py` to see `create_refund`. Three targeted reads have traced the full call chain. No exploratory reads were wasted.

---

**Tool selection table**

| Task | Tool |
|------|------|
| Find all files that import `pandas` | `Grep` |
| Find all `*.test.tsx` files | `Glob` |
| Find where a function is defined | `Grep` |
| Find all callers of a function | `Grep` |
| Load a specific file to review | `Read` |
| Make a targeted change (unique anchor) | `Edit` |
| Make a change when anchor is non-unique | `Read` + `Write` |
| Run tests or shell commands | `Bash` |

**Grep vs. Glob decision:**
- Use `Grep` when you know something about the *content* (a function name, an import, an error string).
- Use `Glob` when you know something about the *filename or extension* (`**/*.config.ts`, `tests/**/*.py`).
- They compose naturally: `Glob` to get a file list, `Grep` within those files to narrow further.

**Key exam facts for 2.5:**
- `Grep` = search file **contents** for patterns. `Glob` = find files by **name/extension**.
- `Edit` requires a **unique** text anchor. When anchor appears multiple times, fall back to `Read` + `Write`.
- Incremental exploration: `Grep` entry points → `Read` to follow imports → never read all files upfront.
- Tracing function usage: identify all exported names first, then `Grep` each name across the codebase.

| Task | Tool |
|------|------|
| Find files importing 'pandas' | `Grep` |
| Find all `*.test.tsx` files | `Glob` |
| Load a file to review | `Read` |
| Fix a typo in a unique location | `Edit` |
| Edit fails (non-unique anchor) | `Read` + `Write` |
| Run test suite | `Bash` |

---
## Domain 2 Capstone Project: Multi-Tool Research Agent with Scoped Tool Access

**Description:** Build a two-agent research system that demonstrates all five task statements:
- **2.1** Well-differentiated tool descriptions prevent misrouting
- **2.2** Structured error responses enable intelligent recovery
- **2.3** Scoped tool access (each agent gets only what it needs) + `tool_choice: "any"` for guaranteed structured output
- **2.4** MCP-style resource manifest reduces exploratory calls
- **2.5** Built-in tool selection reasoning (Grep vs Glob)

**Success criteria:** The search agent uses only its scoped tools, returns structured errors with correct categories, and the synthesis agent uses `verify_fact` for simple checks without coordinator round trips.

In [ ]:
# Domain 2 Capstone: Research Agent with scoped tools, structured errors, guaranteed output

# === Tool Definitions (well-differentiated descriptions) ===

SEARCH_AGENT_TOOLS = [
    {
        "name": "web_search",
        "description": (
            "Searches the web for current information using natural language queries. "
            "Use for recent events, statistics, and factual information not in training data. "
            "Input: a natural language search query. Returns: list of relevant results with titles and summaries. "
            "Do NOT use for searching local files or documents."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"]
        }
    },
    {
        "name": "search_academic_papers",
        "description": (
            "Searches academic databases for peer-reviewed papers and research studies. "
            "Use specifically when the user needs citations, methodology, or peer-reviewed findings. "
            "Do NOT use for general web queries — use web_search for those."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string"},
                "year_from": {"type": "integer", "description": "Filter papers from this year onward"}
            },
            "required": ["query"]
        }
    }
]

SYNTHESIS_AGENT_TOOLS = [
    {
        "name": "verify_fact",
        "description": (
            "Verifies a specific factual claim against provided source text. "
            "Use for simple fact-checks (dates, names, statistics) without needing the coordinator. "
            "Returns: supported | contradicted | not_found, plus the relevant excerpt."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "claim": {"type": "string"},
                "source_text": {"type": "string"}
            },
            "required": ["claim", "source_text"]
        }
    }
]


# === Tool Executors with Structured Error Responses ===

def execute_search_tool(tool_name: str, tool_input: dict) -> str:
    """Stub with structured error responses."""
    if tool_name == "web_search":
        query = tool_input.get("query", "")
        if not query:
            return json.dumps(make_error_response(
                "validation", False,
                "query parameter is required and cannot be empty."
            ))
        if "ERROR" in query.upper():
            return json.dumps(make_error_response(
                "transient", True,
                "Search API timeout. Retry may succeed."
            ))
        return json.dumps({
            "isError": False,
            "results": [
                {"title": f"Result 1 for: {query}", "summary": "AI is transforming creative workflows significantly.", "source": "https://example.com/1"},
                {"title": f"Result 2 for: {query}", "summary": "Creative professionals report 30-40% productivity gains.", "source": "https://example.com/2"}
            ]
        })

    if tool_name == "search_academic_papers":
        return json.dumps({
            "isError": False,
            "papers": [
                {"title": "AI and Creative Industries: A Systematic Review", "year": 2024,
                 "abstract": "This review synthesizes 47 studies on AI's impact on creative professionals.",
                 "doi": "10.1234/example"}
            ]
        })

    return json.dumps(make_error_response("validation", False, f"Unknown tool: {tool_name}"))


def execute_synthesis_tool(tool_name: str, tool_input: dict) -> str:
    if tool_name == "verify_fact":
        claim = tool_input.get("claim", "")
        source = tool_input.get("source_text", "")
        claim_words = set(claim.lower().split())
        source_words = set(source.lower().split())
        overlap = claim_words & source_words
        verdict = "supported" if len(overlap) > 3 else "not_found"
        return json.dumps({"verdict": verdict, "excerpt": source[:100], "claim": claim})
    return json.dumps(make_error_response("validation", False, f"Unknown tool: {tool_name}"))


def run_search_agent(topic: str, max_turns: int = 5) -> dict:
    """Search agent: scoped to web_search and search_academic_papers only."""
    print(f"  [Search Agent] Researching: '{topic}'")
    messages = [{"role": "user", "content": f"Research this topic and gather key findings: {topic}"}]

    for turn in range(max_turns):
        response = client.messages.create(
            model=MODEL, max_tokens=512,
            tools=SEARCH_AGENT_TOOLS,
            messages=messages
        )

        if response.stop_reason == "end_turn":
            return {"findings": " ".join(b.text for b in response.content if hasattr(b, "text")),
                    "source": "web+academic"}

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"    Tool: {block.name}({block.input})")
                    result = execute_search_tool(block.name, block.input)
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

    print(f"  [Search Agent] Reached max_turns ({max_turns}), returning partial findings.")
    last_text = []
    for msg in messages:
        if isinstance(msg.get("content"), list):
            for block in msg["content"]:
                if hasattr(block, "text"):
                    last_text.append(block.text)
    return {"findings": " ".join(last_text) or "Research incomplete after max turns.", "source": "web+academic"}


def run_synthesis_agent(topic: str, search_findings: dict) -> str:
    """Synthesis agent: scoped to verify_fact only. Uses tool_choice: any for guaranteed structured output.

    tool_choice: any forces stop_reason: tool_use on the first turn, so the agent
    must execute the tool and send tool_result blocks before it can return text.
    """
    print(f"  [Synthesis Agent] Synthesizing findings for: '{topic}'")
    messages = [{
        "role": "user",
        "content": f"Topic: {topic}\n\nFindings from search agent:\n{search_findings['findings']}\n\nWrite a 2-3 sentence synthesis."
    }]

    # Turn 1: tool_choice: any forces verify_fact to be called
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        tools=SYNTHESIS_AGENT_TOOLS,
        tool_choice={"type": "any"},
        system="You are a synthesis agent. Use verify_fact to check claims before including them in your summary.",
        messages=messages
    )

    # Execute all tool calls and collect results
    if response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"    Tool: {block.name}({json.dumps(block.input)[:80]}...)")
                result = execute_synthesis_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

        # Turn 2: tool results sent — model produces final text with tool_choice: auto
        final_response = client.messages.create(
            model=MODEL,
            max_tokens=512,
            tools=SYNTHESIS_AGENT_TOOLS,
            tool_choice={"type": "auto"},
            system="You are a synthesis agent. Use verify_fact to check claims before including them in your summary.",
            messages=messages
        )
        return " ".join(b.text for b in final_response.content if hasattr(b, "text"))

    # Fallback: end_turn on first call (tool_choice: any should prevent this, but be safe)
    return " ".join(b.text for b in response.content if hasattr(b, "text"))


# Run the full pipeline
print("=== Domain 2 Capstone: Scoped Multi-Agent Research ===")
topic = "impact of AI on creative industries"

print("\nPhase 1: Search Agent")
search_results = run_search_agent(topic)
print(f"  Findings: {search_results['findings'][:150]}...")

print("\nPhase 2: Synthesis Agent")
synthesis = run_synthesis_agent(topic, search_results)
print(f"\nFinal synthesis:\n{synthesis}")

print("\nOBSERVE:")
print("  - Search agent only called web_search or search_academic_papers (never synthesis tools)")
print("  - Synthesis agent called verify_fact then produced final text (tool_choice: any enforced the call)")
print("  - Structured errors would enable the coordinator to retry or route to fallback")

---
## Domain 2 Complete

**Summary of key exam facts:**

| Task | Core Principle |
|------|----------------|
| 2.1 | Tool descriptions are the primary selection mechanism. Include input format, examples, boundary conditions, and when NOT to use. Split generic tools into purpose-specific ones. |
| 2.2 | Return `isError`, `errorCategory`, `isRetryable`. Four categories: transient, validation, business, permission. Access failure ≠ valid empty result. |
| 2.3 | 4-5 tools max per agent. Scoped cross-role tools for high-frequency needs. `tool_choice: "any"` guarantees a tool call. Forced selection enforces step ordering. |
| 2.4 | `.mcp.json` = project/team. `~/.claude.json` = personal. `${ENV_VAR}` for credentials. Resources = catalogs (reduce exploratory calls). Enhance MCP descriptions vs built-ins. |
| 2.5 | Grep = content search. Glob = file name patterns. Edit requires unique anchor; fallback = Read + Write. Explore incrementally: Grep entry points → Read to follow imports. |